In [32]:
import trafilatura
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
from langchain_ollama import OllamaLLM
from langchain_core.prompts import PromptTemplate
from langchain_community.document_loaders import PyPDFLoader
import tempfile
import requests
import json
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os

# Initialize our Local LLM
llm = OllamaLLM(model="llama3.1", format="json")

# Configuration
KEYWORDS = ["licenta", "master", "doctorat", "calendar", "taxe", "acte", "admitere", "candidati"]
BAD_KEYWORDS = ["anunt", "finalizare", "xls", "teze", "docx", "plan"]
START_URLS = ["https://www.tuiasi.ro/admitere", 
              "https://ac.tuiasi.ro/admitere", 
              "https://arh.tuiasi.ro/admitere-2025/", 
              "https://etti.tuiasi.ro/admitere", 
              "https://icpm.tuiasi.ro/admitere/", 
              "https://ci.tuiasi.ro/admitere/", 
              "https://cmmi.tuiasi.ro/admitere/", 
              "https://hgim.tuiasi.ro/admitere/",  
              "https://ieeia.tuiasi.ro/admitere/", 
              "https://mec.tuiasi.ro/admitere/", 
              "https://sim.tuiasi.ro/admitere/", 
              "https://dima.tuiasi.ro/admitere/"]    


In [21]:
def get_page_links(url):
    """
    Extracts and cleans internal links from a page, filtering only those 
    that contain specific admission-related keywords in the URL string.
    """
    try:
        downloaded = trafilatura.fetch_url(url)
        if not downloaded:
            return []
        
        soup = BeautifulSoup(downloaded, "html.parser")
        absolute_links = set()
        base_domain = urlparse(url).netloc
        
        for a_tag in soup.find_all("a", href=True):
            href = a_tag["href"].lower()
            full_url = urljoin(url, href)
            
            parsed_url = urlparse(full_url)
            clean_link = f"{parsed_url.scheme}://{parsed_url.netloc}{parsed_url.path}".rstrip('/')
            
            # 1. Domain Check: Must stay on the same faculty/university domain
            if base_domain in parsed_url.netloc and parsed_url.scheme in ["http", "https"]:
                
                # 2. Keyword Check: Does the URL contain any of our keywords?
                # We check the 'clean_link' string to see if keywords like 'taxe' or 'licenta' appear
                has_good_word = any(word in clean_link for word in KEYWORDS)
                has_bad_word = any(word in clean_link for word in BAD_KEYWORDS)
                
                if has_good_word and not has_bad_word:
                    absolute_links.add(clean_link)
                
        return list(absolute_links)
    except Exception as e:
        print(f"⚠️ Error fetching links from {url}: {e}")
        return []

In [22]:
queue = [(url, 0) for url in START_URLS]
visited = set()
all_discovered_links = set()

while queue:
    current_url, depth = queue.pop(0)
    
    if current_url in visited:
        continue
    
    visited.add(current_url)
    all_discovered_links.add(current_url)
    
    prefix = "  " * depth + "└─"
    print(f"{prefix} [{depth}] Visiting: {current_url}")
    
    if depth < MAX_DEPTH:
        sublinks = get_page_links(current_url)
        
        new_links_found = 0
        for link in sublinks:
            if link not in visited:
                queue.append((link, depth + 1))
                new_links_found += 1
        
        if new_links_found > 0:
            pass 

with open("urls.txt", "w", encoding="utf-8") as f:
    for link in sorted(all_discovered_links):
        f.write(f"{link}\n")

print("💾 Listsaved in 'urls.txt'.")

└─ [0] Visiting: https://www.tuiasi.ro/admitere
└─ [0] Visiting: https://ac.tuiasi.ro/admitere
└─ [0] Visiting: https://arh.tuiasi.ro/admitere-2025/
└─ [0] Visiting: https://etti.tuiasi.ro/admitere
└─ [0] Visiting: https://icpm.tuiasi.ro/admitere/
└─ [0] Visiting: https://ci.tuiasi.ro/admitere/
└─ [0] Visiting: https://cmmi.tuiasi.ro/admitere/
└─ [0] Visiting: https://hgim.tuiasi.ro/admitere/
└─ [0] Visiting: https://ieeia.tuiasi.ro/admitere/
└─ [0] Visiting: https://mec.tuiasi.ro/admitere/
└─ [0] Visiting: https://sim.tuiasi.ro/admitere/
└─ [0] Visiting: https://dima.tuiasi.ro/admitere/
  └─ [1] Visiting: https://www.tuiasi.ro/licenta
  └─ [1] Visiting: https://www.tuiasi.ro/licenta/metode-de-selectie
  └─ [1] Visiting: https://www.tuiasi.ro/doctorat/documente-utile
  └─ [1] Visiting: https://www.tuiasi.ro/compartimentul-protectia-datelor-cu-caracter-personal
  └─ [1] Visiting: https://www.tuiasi.ro/admitere/master
  └─ [1] Visiting: https://www.tuiasi.ro/licenta/taxe-si-finantare
  └

In [43]:
def get_page_content(url):
    """
    Extracts only the clean text from a page.
    Best for feeding the LLM or building the vector DB.
    """
    try:
        if url.lower().endswith(".pdf"):
            loader = PyPDFLoader(url)
            pages = loader.load()
            return "\n".join([p.page_content for p in pages])
        else:
            downloaded = trafilatura.fetch_url(url)
            return trafilatura.extract(downloaded) if downloaded else None
    except Exception as e:
        print(f"⚠️ Error fetching content from {url}: {e}")
        return None

def get_faculty_tag(url):
    domain = urlparse(url).netloc
    parts = domain.split('.')
    if len(parts) >= 3 and parts[-2] == "tuiasi":
        return parts[0].upper()
    return "TUIASI_CENTRAL"

print(get_page_content("https://ac.tuiasi.ro/admitere"))

MENUMENU
- Despre
-
-
Departamente
- Departamentul de Automatică și Informatică Aplicată
- Departamentul de Calculatoare
Societăți profesionale
- Societatea Română de Automatică și Informatică Tehnică (SRAIT)
-
- Noutăți
- AC35
- Admitere
-
-
Alte informații
-
- Cercetare
- Studii
-
-
-
-
Programe studii licență
-
Programe studii master
-
Alte informații
-
-
-
- Studenți


In [44]:
url_check_prompt = PromptTemplate.from_template("""
Analizează următorul URL de la Universitatea TUIASI: {url}
Anul curent de interes: 2026.

Sarcina: Decide dacă acest link pare să conțină informații esențiale pentru admitere (Acte, Taxe, Calendar, Metodologie, Programe de studiu).
Atenție: Respinge link-urile care conțin "2025", "2024" sau ani anteriori în URL.

Răspunde DOAR JSON:
{{
    "is_important": true/false,
    "topic": "licenta/master/doctorat/general",
    "content_type": "taxe/acte/calendar/metodologie/programe/alta"
}}
""")

extraction_prompt = PromptTemplate.from_template("""
Ești un asistent de admitere TUIASI 2026. Analizează textul brut de mai jos extras de la URL-ul: {url}

Sarcina ta:
1. Verifică dacă textul este relevant pentru admitere sau reguli generale. Informații de interes: 
    - Acte necesare (Enrollment documents)
    - Taxe și finanțare (Tuition fees)
    - Calendarul admiterii (Admission calendar)
    - Reguli / Metodologii / Condiții (Rules and Regulations)
    - Informații despre programele de studiu (Study programs)
2. Împarte textul în "chunk-uri" logice (fragmente de sine stătătoare). Fiecare chunk trebuie să fie un obiect separat.
3. Pentru fiecare chunk, asigură-te că:
    - Include contextul necesar (ex: în loc de "Taxe", scrie "Taxe de școlarizare Facultatea AC 2026").
    - Este formatat curat (păstrează tabelele dacă există).
    - Nu depășește aproximativ 1000 de caractere.
4. Identifică tag-urile relevante pentru fiecare fragment din lista permisă: {KEYWORDS}.
5. Dacă textul este în engleză, adaugă obligatoriu tag-ul "english".

TEXT RAW:
{raw_content}

Răspunde EXCLUSIV sub formă de obiect JSON cu următoarea structură:
{{
    "is_valid_2026": true/false,
    "reason": "Explică pe scurt de ce ai acceptat sau respins acest text",
    "language": "ro/en",
    "chunks": [
        {{
            "content": "Textul primului fragment...",
            "chunk_tags": ["tag1"]
        }}
    ]
}}
""")

In [ ]:
import json
import os
from langchain_core.documents import Document

OUTPUT_FILE = "processed_documents.jsonl"
processed_docs = []

# Helper function to append documents to a file immediately
def save_to_jsonl(doc, file_path):
    with open(file_path, "a", encoding="utf-8") as f:
        json_obj = {
            "page_content": doc.page_content,
            "metadata": doc.metadata
        }
        f.write(json.dumps(json_obj, ensure_ascii=False) + "\n")

with open("urls.txt", "r", encoding="utf-8") as f:
    urls = [line.strip() for line in f.readlines()]

print(f"🚀 Starting processing for {len(urls)} URLs...")

for url in urls:
    faculty = get_faculty_tag(url)
    
    try:
        # Step 1: Check if URL is important
        url_resp = llm.invoke(url_check_prompt.format(url=url))
        url_data = json.loads(url_resp)
        
        if not url_data.get("is_important"):
            print(f"⏩ Ignored (Unimportant URL): {url}")
            continue
            
        # Step 2: Fetch raw content
        raw_text = get_page_content(url)
        
        if not raw_text or len(raw_text) < 200:
            print(f"⚠️ Ignored (Content too short or empty): {url}")
            continue
            
        # Step 3: LLM Semantic Chunking & Extraction
        # Note: Added a safety limit [:12000] so huge PDFs don't break the context window
        extract_resp = llm.invoke(extraction_prompt.format(
            url=url, 
            raw_content=raw_text[:12000],
            KEYWORDS=KEYWORDS 
        ))
        extract_data = json.loads(extract_resp)
        
        # Check if relevant for 2026 and if the LLM successfully created chunks
        if extract_data.get("is_valid_2026") and "chunks" in extract_data:
            lang_tag = extract_data.get("language", "ro")
            chunks_count = 0
            
            for chunk_obj in extract_data["chunks"]:
                # Combine tags from the URL check and the specific chunk
                tags = [faculty, url_data.get("topic"), url_data.get("content_type")]
                tags.extend(chunk_obj.get("chunk_tags", []))
                
                # Ensure english tag is present if language is en
                if lang_tag == "en" or "english" in str(tags).lower():
                    tags.append("english")
                
                # Clean tags (remove None, empty strings, and duplicates)
                clean_tags = list(set([str(t).lower() for t in tags if t and str(t).strip() != ""]))
                
                # Construct the final content by prepending tags and source
                tag_header = f"TAGS: {', '.join(clean_tags)}\nSOURCE: {url}\n\n"
                full_content = tag_header + chunk_obj.get("content", "")
                
                # Create the Document object
                doc = Document(
                    page_content=full_content,
                    metadata={
                        "source": url,
                        "faculty": faculty,
                        "topic": url_data.get("topic"),
                        "lang": lang_tag,
                        "year": "2026"
                    }
                )
                
                # Store in memory and save to disk immediately
                processed_docs.append(doc)
                save_to_jsonl(doc, OUTPUT_FILE)
                chunks_count += 1
                
            print(f"   ✅ Successfully added and saved ({chunks_count} semantic chunks) {url}")
        else:
            motiv = extract_data.get('reason', 'No reason offered by the LLM')
            print(f"   ❌ Rejected: {url} | Motiv: {motiv}")

    except Exception as e:
        print(f"⚠️ Error at url: {url}: {e}")

print(f"\n✨ Done! Created and saved {len(processed_docs)} docs in '{OUTPUT_FILE}'.")

🚀 Starting processing for 414 URLs...
